# ViT 518 con pesos de fondo de ojo (RETFound-DINOv2) — sup_k12 + sano (13 etiquetas)

Versión fundus del run `vit518asl`: misma receta que `vit_dinov2_sup_k12_sano_hires518_asl` (18_train_hires_asl.py, ViT-L/14 a 518 px, ASL, 20 épocas, batch 4 × accum 8, lr 1e-3 / backbone 2e-5, 4 bloques, patience 6) pero inicializada con RETFound. Si da error/OOM reintenta con 2 bloques, batch 2 × accum 16.

Control = `vit_dinov2_sup_k12_sano_hires518_asl` (pesos DINOv2 genéricos).

## Antes de ejecutar (una vez)

1. **Acceso RETFound**: acepta el formulario en https://huggingface.co/YukunZhou/RETFound_dinov2_meh y crea un token de *lectura*.
2. **Secretos de Colab** (icono 🔑, activa *Notebook access*): `HF_TOKEN`, y `KAGGLE_API_TOKEN` (KGAT_...) **o** `KAGGLE_USERNAME` + `KAGGLE_KEY`.
3. **Código**: en tu PC ejecuta `python training/dual_tax/colab/make_colab_bundle_fundus.py` y sube
   `retina_vit_fundus_colab_bundle.zip` a `MyDrive/retina_vit_fundus/code/`.
4. **Controles v2** (opcional, para la tabla final): copia las carpetas de los runs de control (basta con
   `val_predictions*.npz` y `test_predictions*.npz`) a `MyDrive/retina_vit_fundus/controls/<nombre_del_run>/`.
5. Entorno → Cambiar tipo de entorno → **GPU** (A100 o L4 mejor que T4). Después: *Ejecutar todo*.

**Reanudar tras desconexión**: vuelve a *Ejecutar todo*. Los runs terminados están en Drive y se saltan; el run que
se cortó se reinicia desde la época 1 (los scripts de entrenamiento no reanudan por época). Mientras entrena, cada
10 min se copia el mejor checkpoint parcial a `runs/<TAG>/<run>__partial/` como respaldo.


In [ ]:
# ==== Parámetros ====
TAG = "supk12"
RUN = "vit_dinov2_sup_k12_sano_hires518_asl_fundus"
CONTROL = "vit_dinov2_sup_k12_sano_hires518_asl"
IMG_SIZE = 518
EPOCHS, PATIENCE, UNFREEZE = 20, 6, 4
BATCH, ACCUM = 4, 8            # lote efectivo 32. A100: BATCH, ACCUM = 8, 4. T4 con OOM: 2, 16
HEAD_LR, BACKBONE_LR = "1e-3", "2e-5"


In [ ]:
# ==== 0. Drive + secretos ====
from pathlib import Path
import os, sys, time, json, shutil, zipfile, subprocess, threading

from google.colab import drive, userdata
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/retina_vit_fundus")
DRIVE_CODE_ZIP = DRIVE_ROOT / "code" / "retina_vit_fundus_colab_bundle.zip"
DRIVE_RUNS = DRIVE_ROOT / "runs" / TAG          # salidas + checkpoints de este cuaderno
DRIVE_CONTROLS = DRIVE_ROOT / "controls"         # runs v2 de control (val/test_predictions*.npz)
DRIVE_CACHE = DRIVE_ROOT / "cache"               # caché HF/torch (RETFound/DINOv2) persistente
for d in (DRIVE_ROOT / "code", DRIVE_RUNS, DRIVE_CONTROLS, DRIVE_CACHE):
    d.mkdir(parents=True, exist_ok=True)

def _secret(name, required=True):
    try:
        v = userdata.get(name)
    except Exception:
        v = None
    if required and not v:
        raise SystemExit(f"Falta el secreto de Colab '{name}' (icono de la llave, activa 'Notebook access').")
    return (v or "").strip()

os.environ["HF_TOKEN"] = _secret("HF_TOKEN")                  # token de lectura HF con acceso a RETFound_dinov2_meh
_kgat = _secret("KAGGLE_API_TOKEN", required=False)           # KGAT_... (o bien KAGGLE_USERNAME + KAGGLE_KEY)
if _kgat:
    os.environ["KAGGLE_API_TOKEN"] = _kgat
else:
    os.environ["KAGGLE_USERNAME"] = _secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = _secret("KAGGLE_KEY")

os.environ["VIT_PRETRAINED"] = "fundus"                       # RETFound-DINOv2 (lo aplica la celda 1b, parche runtime)
os.environ["HF_HOME"] = str(DRIVE_CACHE / "hf")
os.environ["HUGGINGFACE_HUB_CACHE"] = str(DRIVE_CACHE / "hf" / "hub")
os.environ["TORCH_HOME"] = str(DRIVE_CACHE / "torch")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NINGUNA (Entorno > Cambiar tipo de entorno)")


In [ ]:
# ==== 1. Código + datos RFMiD ====
PROJECT_ROOT = Path("/content/retina_multilabel")
assert DRIVE_CODE_ZIP.is_file(), f"Sube el bundle a {DRIVE_CODE_ZIP} (ver celda de instrucciones)"
if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
PROJECT_ROOT.mkdir(parents=True)
with zipfile.ZipFile(DRIVE_CODE_ZIP) as zf:
    zf.extractall(PROJECT_ROOT)
assert (PROJECT_ROOT / "src" / "retina_multilabel" / "model.py").is_file()

%pip install -q "timm>=1.0" scikit-learn pandas pillow tqdm huggingface_hub transformers kagglehub

import kagglehub
dl = Path(kagglehub.dataset_download("andrewmvd/retinal-disease-classification"))

def _find_rfmid(root: Path) -> Path:
    for labels in root.rglob("RFMiD_Training_Labels.csv"):
        for parent in [labels.parent, *labels.parents]:
            if all((parent / s).is_dir() for s in ("Training_Set", "Evaluation_Set", "Test_Set")):
                return parent
    raise FileNotFoundError(f"RFMiD incompleto bajo {root}")

DATA_ROOT = _find_rfmid(dl)
os.environ["RETINA_DATA_ROOT"] = str(DATA_ROOT)
os.environ["PYTHONPATH"] = os.pathsep.join([str(PROJECT_ROOT / "src"), os.environ.get("PYTHONPATH", "")])
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("DATA_ROOT", DATA_ROOT)

# Comprobación rápida: RETFound accesible con el token (falla aquí y no a mitad de entrenamiento)
from huggingface_hub import HfApi
HfApi().model_info("YukunZhou/RETFound_dinov2_meh", token=os.environ["HF_TOKEN"])
print("RETFound_dinov2_meh accesible")


In [ ]:
# ==== 1b. Parche RETFound (runtime, sin tocar el repo) ====
# Envuelve retina_multilabel.model._build_dinov2_backbone: si VIT_PRETRAINED=fundus carga RETFound_dinov2_meh
# (quita prefijos teacher/module./backbone./encoder., w12->fc1, w3->fc2, interpola pos_embed, strict=False,
# registra claves y exige >=90% cargadas). Se escribe en /content/fundus_patch y llega a los subprocesos
# de entrenamiento vía sitecustomize.py en PYTHONPATH.
PATCH_DIR = Path('/content/fundus_patch'); PATCH_DIR.mkdir(parents=True, exist_ok=True)
(PATCH_DIR / 'fundus_vit_patch.py').write_text('"""RETFound-DINOv2 init for the DINOv2 ViT-L backbone, applied at runtime (no repo files touched).\n\nWraps retina_multilabel.model._build_dinov2_backbone when VIT_PRETRAINED=fundus.\nLoaded in every Python subprocess through sitecustomize.py on PYTHONPATH.\n"""\nimport os\n\nVIT_FUNDUS_REPO = "YukunZhou/RETFound_dinov2_meh"\nVIT_FUNDUS_FILE = "RETFound_dinov2_meh.pth"\n\n\ndef clean_retfound_state_dict(ckpt):\n    sd = ckpt\n    for key in ("teacher", "model", "state_dict"):\n        if isinstance(sd, dict) and key in sd and isinstance(sd[key], dict):\n            sd = sd[key]\n            break\n    clean = {}\n    for k, v in sd.items():\n        for pre in ("module.", "backbone.", "encoder."):\n            if k.startswith(pre):\n                k = k[len(pre):]\n        k = k.replace("mlp.w12.", "mlp.fc1.").replace("mlp.w3.", "mlp.fc2.")\n        clean[k] = v\n    return clean\n\n\ndef load_fundus_vit(backbone, path=None, min_frac=0.9):\n    import torch\n    from timm.layers import resample_abs_pos_embed\n\n    path = path or os.environ.get("VIT_FUNDUS_WEIGHTS")\n    if not path:\n        token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")\n        if not token:\n            raise RuntimeError(f"VIT_PRETRAINED=fundus needs HF_TOKEN (gated {VIT_FUNDUS_REPO}) or VIT_FUNDUS_WEIGHTS")\n        from huggingface_hub import hf_hub_download\n        path = hf_hub_download(VIT_FUNDUS_REPO, VIT_FUNDUS_FILE, token=token)\n    clean = clean_retfound_state_dict(torch.load(path, map_location="cpu", weights_only=False))\n    own = backbone.state_dict()\n    if "pos_embed" in clean and "pos_embed" in own and clean["pos_embed"].shape != own["pos_embed"].shape:\n        npt = 0 if getattr(backbone, "no_embed_class", False) else getattr(backbone, "num_prefix_tokens", 1)\n        old = tuple(clean["pos_embed"].shape)\n        clean["pos_embed"] = resample_abs_pos_embed(\n            clean["pos_embed"], new_size=list(backbone.patch_embed.grid_size), num_prefix_tokens=npt)\n        print(f"[fundus] pos_embed resampled {old} -> {tuple(clean[\'pos_embed\'].shape)}", flush=True)\n    shape_skip = [k for k, v in clean.items() if k in own and own[k].shape != v.shape]\n    for k in shape_skip:\n        clean.pop(k)\n    res = backbone.load_state_dict(clean, strict=False)\n    loaded = len([k for k in clean if k in own])\n    print(f"[fundus] {path}: loaded={loaded}/{len(own)} missing={res.missing_keys} "\n          f"unexpected={len(res.unexpected_keys)} {res.unexpected_keys[:8]} shape_skipped={shape_skip}", flush=True)\n    if loaded < min_frac * len(own):\n        raise RuntimeError(f"RETFound checkpoint matched only {loaded}/{len(own)} keys")\n    return {"loaded": loaded, "total": len(own), "missing": res.missing_keys, "unexpected": res.unexpected_keys}\n\n\ndef apply_patch(mod=None):\n    if mod is None:\n        import retina_multilabel.model as mod\n    orig = mod._build_dinov2_backbone\n    if getattr(orig, "_fundus_patched", False):\n        return\n    def _build_dinov2_backbone(model_config):\n        bundle = orig(model_config)\n        if os.environ.get("VIT_PRETRAINED", "default").strip().lower() == "fundus":\n            load_fundus_vit(bundle.backbone)   # requires_grad flags are kept by load_state_dict\n            bundle.description += " + RETFound-DINOv2 fundus init (notebook patch)"\n        return bundle\n    _build_dinov2_backbone._fundus_patched = True\n    mod._build_dinov2_backbone = _build_dinov2_backbone\n    print("[fundus] patched retina_multilabel.model._build_dinov2_backbone "\n          f"(VIT_PRETRAINED={os.environ.get(\'VIT_PRETRAINED\', \'default\')})", flush=True)\n')
(PATCH_DIR / 'sitecustomize.py').write_text('# Auto-generado por el cuaderno (runtime, fuera del repo): aplica el parche RETFound en cada subproceso Python.\nimport importlib.abc, importlib.util, os, sys\n\n_here = os.path.dirname(os.path.abspath(__file__))\n# encadenar el sitecustomize original de Colab, si existe\nfor _p in sys.path:\n    _f = os.path.join(_p, "sitecustomize.py")\n    if _p and os.path.abspath(_p) != _here and os.path.isfile(_f):\n        try:\n            exec(compile(open(_f).read(), _f, "exec"), {"__name__": "_orig_sitecustomize", "__file__": _f})\n        except Exception as _e:\n            print(f"[fundus] sitecustomize original falló: {_e}", file=sys.stderr)\n        break\n\n_TARGET = "retina_multilabel.model"\n\nclass _FundusFinder(importlib.abc.MetaPathFinder):\n    def find_spec(self, name, path, target=None):\n        if name != _TARGET:\n            return None\n        sys.meta_path.remove(self)\n        try:\n            spec = importlib.util.find_spec(name)\n        finally:\n            sys.meta_path.insert(0, self)\n        if spec is None or spec.loader is None:\n            return spec\n        orig_exec = spec.loader.exec_module\n        def exec_module(module):\n            orig_exec(module)\n            import fundus_vit_patch\n            fundus_vit_patch.apply_patch(module)\n        spec.loader.exec_module = exec_module\n        return spec\n\nif not any(isinstance(f, _FundusFinder) for f in sys.meta_path):\n    sys.meta_path.insert(0, _FundusFinder())\n')
os.environ['PYTHONPATH'] = os.pathsep.join([str(PATCH_DIR), os.environ.get('PYTHONPATH', '')])
sys.path.insert(0, str(PATCH_DIR))
import fundus_vit_patch, retina_multilabel.model as _rm
fundus_vit_patch.apply_patch(_rm)   # también en el kernel (EVAL / pruebas)
_chk = subprocess.run([sys.executable, '-c', 'import retina_multilabel.model as m; assert getattr(m._build_dinov2_backbone, "_fundus_patched", False); print("subproceso parcheado OK")'],
                      env=os.environ.copy(), capture_output=True, text=True)
print(_chk.stdout, _chk.stderr[-2000:]); assert _chk.returncode == 0, 'el parche no llega a los subprocesos'


In [ ]:
EXP = PROJECT_ROOT / "experiments" / "paper_compare_latent_clusters"
LOCAL_RUNS = EXP / "outputs" / "runs"
# ==== 2. Helpers: ejecutar, sincronizar con Drive, reanudar ====
def _copy_tree(src: Path, dst: Path):
    for p in src.rglob("*"):
        if p.is_file() and "image_cache" not in p.parts:
            out = dst / p.relative_to(src)
            if not out.exists() or out.stat().st_mtime < p.stat().st_mtime or out.stat().st_size != p.stat().st_size:
                out.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(p, out)

def restore_runs():
    """Trae a local los runs ya guardados en Drive (los terminados se saltan)."""
    LOCAL_RUNS.mkdir(parents=True, exist_ok=True)
    for d in DRIVE_RUNS.iterdir():
        if d.is_dir() and (d / "run_summary.json").is_file():
            _copy_tree(d, LOCAL_RUNS / d.name)

def done(run_name):
    return (DRIVE_RUNS / run_name / "run_summary.json").is_file()

def run_job(script: Path, run_name: str, args: list[str]):
    if done(run_name):
        print("SKIP (ya en Drive):", run_name); return
    # run sin terminar de una sesión anterior: se reinicia desde cero (los scripts no reanudan por época)
    shutil.rmtree(DRIVE_RUNS / run_name, ignore_errors=True)
    local = LOCAL_RUNS / run_name
    stop = threading.Event()
    def _bg():  # copia el mejor checkpoint/historial a Drive cada 10 min
        while not stop.wait(600):
            if local.is_dir():
                _copy_tree(local, DRIVE_RUNS / (run_name + "__partial"))
    th = threading.Thread(target=_bg, daemon=True); th.start()
    cmd = [sys.executable, str(script), "--run-name", run_name, *args]
    print(">>>", " ".join(cmd), flush=True)
    t0 = time.time()
    rc = subprocess.run(cmd, cwd=str(PROJECT_ROOT)).returncode
    stop.set(); th.join(timeout=5)
    print(f"<<< rc={rc} en {(time.time()-t0)/3600:.2f} h", flush=True)
    if rc == 0 and (local / "run_summary.json").is_file():
        _copy_tree(local, DRIVE_RUNS / run_name)
        shutil.rmtree(DRIVE_RUNS / (run_name + "__partial"), ignore_errors=True)
        print("[Drive]", DRIVE_RUNS / run_name)
    else:
        raise RuntimeError(f"{run_name} falló (rc={rc}); revisa el log. Si es OOM baja BATCH_* en la celda de parámetros.")

sys.path.insert(0, str(EXP / "src"))
os.environ["PYTHONPATH"] = os.pathsep.join([str(EXP / "src"), os.environ["PYTHONPATH"]])
TAX_DIR = EXP / "outputs" / "supervised_k12_sano"
if not (TAX_DIR / "cluster_map.json").is_file():
    subprocess.run([sys.executable, str(EXP / "scripts" / "01g_build_supervised_k12_sano.py")], cwd=str(PROJECT_ROOT), check=True)
assert (TAX_DIR / "labels" / "clustered_train.csv").is_file()
_copy_tree(TAX_DIR, DRIVE_RUNS / "supervised_k12_sano")
restore_runs()


## Entrenamiento (fundus, 518 px)

In [ ]:
common = ["--cluster-map", str(TAX_DIR / "cluster_map.json"), "--labels-dir", str(TAX_DIR / "labels"), "--num-workers", "2"]
def args(unfreeze, batch, accum):
    return [*common, "--model", "vit", "--img-size", str(IMG_SIZE), "--epochs", str(EPOCHS),
            "--batch-size", str(batch), "--accum", str(accum), "--lr", HEAD_LR, "--backbone-lr", BACKBONE_LR,
            "--unfreeze-blocks", str(unfreeze), "--patience", str(PATIENCE)]
os.environ["VIT_PRETRAINED"] = "fundus"
try:
    run_job(EXP / "scripts" / "18_train_hires_asl.py", RUN, args(UNFREEZE, BATCH, ACCUM))
except RuntimeError as e:
    print(e, "\nReintento como en Kaggle: 2 bloques, batch 2 x accum 16")
    run_job(EXP / "scripts" / "18_train_hires_asl.py", RUN, args(2, 2, 16))
# Nota: 18_train_hires_asl.py usa AMP internamente; también guarda hires_asl_eval.json (umbrales fbeta1.5/youden/specfloor ajustados en VAL)
print(json.dumps({k: v["test"] for k, v in json.loads((DRIVE_RUNS / RUN / "hires_asl_eval.json").read_text())["results"].items()
                  if k.startswith("tta|")}, indent=1)[:3000])


## Resultados: fundus vs control

Copia la carpeta del control `vit_dinov2_sup_k12_sano_hires518_asl` (sus `val/test_predictions_tta.npz`) a `controls/`. Además de recall≥0.80 y F2, se muestra spec≥0.80 en `hires_asl_eval.json` como en la tabla de 3 semillas.

In [ ]:
PRED_FILES = ["{split}_predictions_tta.npz", "{split}_predictions_notta.npz", "{split}_predictions.npz"]
CRITERIA = ["recall_floor_0.80", "fbeta2", "fbeta1.5"]
PAIRS = [(RUN, CONTROL)]
# ==== Métricas en TEST con umbrales ajustados en VAL (fundus vs control v2) ====
import numpy as np, pandas as pd
from sklearn.metrics import precision_recall_curve, roc_auc_score

def _load(run_dir: Path, split: str):
    for name in PRED_FILES:
        f = run_dir / name.format(split=split)
        if f.is_file():
            z = np.load(f, allow_pickle=False)
            y = np.asarray(z["targets"], dtype=np.float32)
            p = np.asarray(z["probabilities"], dtype=np.float64)
            m = np.asarray(z["active_mask"], bool) if "active_mask" in z.files else np.ones(y.shape[1], bool)
            return y, p, m
    return None

def _thr_fbeta(y, s, beta):
    if y.sum() == 0 or y.sum() == len(y):
        return 0.5
    P, R, T = precision_recall_curve(y, s); P, R = P[:-1], R[:-1]
    b2 = beta * beta
    f = (1 + b2) * P * R / (b2 * P + R + 1e-12)
    return float(T[np.flatnonzero(np.isclose(f, np.nanmax(f)))[-1]])

def _thr_recall_floor(y, s, floor):
    if y.sum() == 0 or y.sum() == len(y):
        return 0.5
    P, R, T = precision_recall_curve(y, s); P, R = P[:-1], R[:-1]
    f1 = 2 * P * R / (P + R + 1e-12)
    ok = np.flatnonzero(R >= floor)
    return float(T[ok[np.argmax(f1[ok])]]) if ok.size else 0.5

def evaluate(run_dir: Path, crit: str):
    val, test = _load(run_dir, "val"), _load(run_dir, "test")
    if val is None or test is None:
        return None
    yv, pv, m = val; yt, pt, mt = test
    m = m & mt & (yt.sum(0) > 0) & (yv.sum(0) > 0)
    rows = []
    for j in np.flatnonzero(m):
        if crit.startswith("fbeta"):
            t = _thr_fbeta(yv[:, j], pv[:, j], float(crit[5:]))
        else:
            t = _thr_recall_floor(yv[:, j], pv[:, j], float(crit.split("_")[-1]))
        yp = pt[:, j] >= t; y = yt[:, j] > 0.5
        tp = (yp & y).sum(); fp = (yp & ~y).sum(); fn = (~yp & y).sum()
        r = tp / max(tp + fn, 1); p = tp / max(tp + fp, 1)
        rows.append(dict(recall=r, precision=p, f1=2*p*r/(p+r+1e-12), f2=5*p*r/(4*p+r+1e-12),
                         auroc=roc_auc_score(y, pt[:, j])))
    d = pd.DataFrame(rows).mean()
    return dict(n_labels=int(m.sum()), **{k: round(float(v), 4) for k, v in d.items()})

def find_control(name: str):
    for root in (DRIVE_CONTROLS, DRIVE_RUNS):
        for cand in [root / name, *root.rglob(name)]:
            if cand.is_dir() and _load(cand, "val") is not None:
                return cand
    return None

rows = []
for fundus_run, control_run in PAIRS:
    for crit in CRITERIA:
        for kind, rd in (("fundus", DRIVE_RUNS / fundus_run), ("control", find_control(control_run))):
            r = evaluate(rd, crit) if rd is not None else None
            rows.append(dict(pair=fundus_run, weights=kind, run=rd.name if rd else f"{control_run} (NO ENCONTRADO)",
                             thresholds=f"VAL:{crit}", **(r or {})))
res = pd.DataFrame(rows)
out_csv = DRIVE_RUNS / f"fundus_vs_control_{TAG}.csv"
res.to_csv(out_csv, index=False)
print("Métricas medidas en TEST; umbrales por etiqueta ajustados en VAL y aplicados tal cual.")
print("Guardado:", out_csv)
res

# Punto de operación del paper (spec >= 0.80, umbrales en VAL) desde hires_asl_eval.json
for name, rd in (("fundus", DRIVE_RUNS / RUN), ("control", find_control(CONTROL))):
    f = rd / "hires_asl_eval.json" if rd else None
    if f and f.is_file():
        print(name, "tta|specfloor0.80 TEST:", json.loads(f.read_text())["results"]["tta|specfloor0.80"]["test"])
res
